# Grain Growth Tracking — Video Analysis

---
### Requirements
```bash
pip install torch torchvision
pip install git+https://github.com/facebookresearch/sam2.git
pip install opencv-python numpy pandas matplotlib scipy ipywidgets
```


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.spatial.distance import cdist
import os
import torch

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

%matplotlib inline
print("All imports OK")
print(f"PyTorch: {torch.__version__}")


---
## 1 · Inputs


In [ ]:
# ── USER INPUTS ─────────────────────────────────────────────────────────────

VIDEO_PATH      = "your_video.mp4"         # <-- path to POM video
MM_PER_PIXEL    = 0.000423                 # <-- scale factor (mm / pixel)

SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_tiny.pt"
SAM2_CONFIG     = "configs/sam2.1/sam2.1_hiera_tiny.yaml"

# ── FRAME SAMPLING ───────────────────────────────────────────────────────────
# For real-time video, this program doesnt sample evey frame
# SAMPLE_EVERY_N_SECONDS: extract one frame every N seconds of video
SAMPLE_EVERY_N_SECONDS = 5.0   # e.g. 5.0 = one frame per 5 seconds

# ── PRE-PROCESSING ───────────────────────────────────────────────────────────
GAUSSIAN_KSIZE  = (5, 5)
GAUSSIAN_SIGMA  = 1.0
MEDIAN_KSIZE    = 5

# ── SEGMENTATION ─────────────────────────────────────────────────────────────
MIN_GRAIN_AREA_PX = 200

# ── GRAIN MATCHING ───────────────────────────────────────────────────────────
# Max centroid displacement (px) between frames to consider same grain
MAX_MATCH_DIST_PX = 50


---
## 2 · Inspect Video


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open video: {VIDEO_PATH}"

fps        = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_s = tot_frames / fps
w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

# Frame indices to sample
sample_interval_frames = max(1, int(fps * SAMPLE_EVERY_N_SECONDS))
sample_frame_indices   = list(range(0, tot_frames, sample_interval_frames))
sample_timestamps_s    = [idx / fps for idx in sample_frame_indices]

print(f"Video       : {w} × {h} px  |  {fps:.2f} fps  |  {duration_s:.1f} s  |  {tot_frames} frames")
print(f"Physical    : {w*MM_PER_PIXEL:.3f} × {h*MM_PER_PIXEL:.3f} mm")
print(f"Sampling    : every {SAMPLE_EVERY_N_SECONDS} s → {len(sample_frame_indices)} frames to process")


---
## 3 · Preview First Frame


In [ ]:
def read_frame(video_path, frame_index):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read frame {frame_index}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

first_frame = read_frame(VIDEO_PATH, 0)

plt.figure(figsize=(7, 5))
plt.imshow(first_frame)
plt.title("First Frame — Raw POM")
plt.axis("off")
plt.tight_layout()
plt.show()


---
## 4 · Load SAM 2


In [ ]:
device = torch.device("cpu")
print(f"Loading SAM 2 on {device} ...")

sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=32,
    pred_iou_thresh=0.80,
    stability_score_thresh=0.90,
    box_nms_thresh=0.7,
    min_mask_region_area=MIN_GRAIN_AREA_PX,
)

print("SAM 2 loaded ✓")


---
## 5 · Helper Functions


In [ ]:
def preprocess(image_rgb):
    bgr     = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    blurred = cv2.GaussianBlur(bgr, GAUSSIAN_KSIZE, GAUSSIAN_SIGMA)
    blurred = cv2.medianBlur(blurred, MEDIAN_KSIZE)
    return cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)


def segment_frame(image_rgb):
    """Run SAM 2 and return list of grain dicts with area + centroid."""
    masks = mask_generator.generate(image_rgb)
    grains = []
    for m in masks:
        binary   = m["segmentation"].astype(np.uint8)
        area_px  = int(binary.sum())
        if area_px < MIN_GRAIN_AREA_PX:
            continue
        cy, cx = ndimage.center_of_mass(binary)
        grains.append({
            "area_px":           area_px,
            "area_mm2":          round(area_px * MM_PER_PIXEL**2, 6),
            "equiv_diameter_um": round(2.0 * np.sqrt(area_px / np.pi) * MM_PER_PIXEL * 1000, 2),
            "centroid_x":        round(cx, 1),
            "centroid_y":        round(cy, 1),
            "mask":              binary,
        })
    return grains


def match_grains(prev_grains, curr_grains, max_dist=MAX_MATCH_DIST_PX):
    """
    Match current grains to previous grains by nearest centroid.
    Returns curr_grains with a 'track_id' assigned.
    Unmatched grains get a new unique track_id.
    """
    if not prev_grains:
        for i, g in enumerate(curr_grains):
            g["track_id"] = i
        return curr_grains

    prev_cents = np.array([[g["centroid_x"], g["centroid_y"]] for g in prev_grains])
    curr_cents = np.array([[g["centroid_x"], g["centroid_y"]] for g in curr_grains])
    prev_ids   = [g["track_id"] for g in prev_grains]

    dist_matrix = cdist(curr_cents, prev_cents)   # shape: (n_curr, n_prev)
    next_new_id = max(prev_ids) + 1

    used_prev = set()
    for i, g in enumerate(curr_grains):
        best_j   = int(np.argmin(dist_matrix[i]))
        best_dist = dist_matrix[i, best_j]
        if best_dist <= max_dist and best_j not in used_prev:
            g["track_id"] = prev_ids[best_j]
            used_prev.add(best_j)
        else:
            g["track_id"] = next_new_id
            next_new_id  += 1

    return curr_grains


def colorize_masks(image_rgb, grains, track_ids=None):
    """Draw coloured grain overlays; colour is fixed per track_id if provided."""
    overlay     = image_rgb.copy().astype(np.float32) / 255.0
    color_layer = np.zeros_like(overlay)
    rng = np.random.default_rng(42)

    color_map = {}
    for g in grains:
        tid = g.get("track_id", id(g))
        if tid not in color_map:
            color_map[tid] = rng.random(3)
        color = color_map[tid]
        for c in range(3):
            color_layer[:, :, c] += g["mask"] * color[c]

    blended = np.clip(0.55 * overlay + 0.45 * color_layer, 0, 1)
    return (blended * 255).astype(np.uint8)

print("Helper functions defined ✓")


---
## 6 · Process Frames & Track Grains

This is the slow step — SAM 2 runs once per sampled frame.  
Progress is printed after each frame.


In [ ]:
all_records  = []    # flat list of dicts for CSV
prev_grains  = []    # grains from previous frame (for matching)
overlays     = []    # store a few overlays for visualisation

for loop_i, (frame_idx, timestamp) in enumerate(zip(sample_frame_indices, sample_timestamps_s)):
    print(f"[{loop_i+1}/{len(sample_frame_indices)}] t={timestamp:.1f}s  (frame {frame_idx})", end=" ... ")

    # Read & pre-process
    frame_rgb  = read_frame(VIDEO_PATH, frame_idx)
    processed  = preprocess(frame_rgb)

    # Segment
    grains = segment_frame(processed)

    # Match to previous frame
    grains = match_grains(prev_grains, grains)

    # Store records
    for g in grains:
        all_records.append({
            "frame_index":       frame_idx,
            "timestamp_s":       round(timestamp, 3),
            "track_id":          g["track_id"],
            "area_px":           g["area_px"],
            "area_mm2":          g["area_mm2"],
            "equiv_diameter_um": g["equiv_diameter_um"],
            "centroid_x":        g["centroid_x"],
            "centroid_y":        g["centroid_y"],
        })

    # Save overlay for first, middle, last frame
    if loop_i in [0, len(sample_frame_indices)//2, len(sample_frame_indices)-1]:
        overlays.append((timestamp, colorize_masks(frame_rgb, grains)))

    prev_grains = grains
    print(f"{len(grains)} grains")

df = pd.DataFrame(all_records)
print(f"\nDone. {df['track_id'].nunique()} unique grain tracks across {len(sample_frame_indices)} frames.")
df.head(10)


---
## 7 · Visualise Segmentation at Key Frames


In [ ]:
fig, axes = plt.subplots(1, len(overlays), figsize=(7 * len(overlays), 6))
if len(overlays) == 1:
    axes = [axes]

for ax, (ts, ov) in zip(axes, overlays):
    ax.imshow(ov)
    ax.set_title(f"t = {ts:.1f} s")
    ax.axis("off")

plt.suptitle("Grain Segmentation — Key Frames", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 8 · Grain Growth Curves

Plot equivalent diameter over time for grains that persist across most frames.


In [ ]:
# Only plot grains seen in at least 50% of frames
n_frames      = df["timestamp_s"].nunique()
min_obs       = max(2, int(n_frames * 0.5))
persistent_ids = (
    df.groupby("track_id")["timestamp_s"]
    .nunique()
    .loc[lambda s: s >= min_obs]
    .index.tolist()
)

print(f"Grains with ≥{min_obs} observations: {len(persistent_ids)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual growth curves (capped at 30)
for tid in persistent_ids[:30]:
    sub = df[df["track_id"] == tid].sort_values("timestamp_s")
    axes[0].plot(sub["timestamp_s"], sub["equiv_diameter_um"],
                 alpha=0.5, linewidth=1)

axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Equivalent Diameter (µm)")
axes[0].set_title("Individual Grain Growth Curves")

# Mean + std envelope
grouped = df[df["track_id"].isin(persistent_ids)].groupby("timestamp_s")["equiv_diameter_um"]
mean_d  = grouped.mean()
std_d   = grouped.std()

axes[1].fill_between(mean_d.index, mean_d - std_d, mean_d + std_d, alpha=0.25, color="#4C8EDA")
axes[1].plot(mean_d.index, mean_d, color="#4C8EDA", linewidth=2, label="Mean diameter")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Equivalent Diameter (µm)")
axes[1].set_title("Mean Grain Size ± 1 Std Dev")
axes[1].legend()

plt.suptitle("Grain Growth Over Time", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 9 · Summary Statistics per Frame


In [ ]:
summary = df.groupby("timestamp_s")["equiv_diameter_um"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max",
).round(2)

print(summary.to_string())


---
## 10 · Save Output CSV


In [ ]:
base     = os.path.splitext(VIDEO_PATH)[0]
csv_path = base + "_grain_growth.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")
print(f"Columns   : {list(df.columns)}")
print(f"Rows      : {len(df)}")

# Also save per-frame summary
summary_path = base + "_frame_summary.csv"
summary.to_csv(summary_path)
print(f"Summary CSV saved → {summary_path}")


---
## Tips for Tuning

| Parameter | Default | Remarks |
|---|---|---|
| `SAMPLE_EVERY_N_SECONDS` | 5.0 | Grains change quickly → lower; video is long → raise |
| `MAX_MATCH_DIST_PX` | 50 | Grains drift a lot between frames → raise; false matches → lower |
| `MIN_GRAIN_AREA_PX` | 200 | Too many tiny artefacts tracked → raise |
| `points_per_side` | 32 | Small grains being missed → raise (slower) |

### CSV Output Columns
| Column | Description |
|---|---|
| `frame_index` | Raw frame number in the video |
| `timestamp_s` | Time in seconds |
| `track_id` | Consistent grain ID across frames |
| `area_px` / `area_mm2` | Grain area |
| `equiv_diameter_um` | Equivalent circular diameter (µm) |
| `centroid_x/y` | Grain centroid position (px) |
